In [ ]:
import random
import time
import tracemalloc
import sys

# Increase recursion limit because intentionally poor Quick Sort
# partitions can create deep recursion on sorted datasets.
sys.setrecursionlimit(20000)


# ---------------------------------------------------------
# MERGE SORT
# ---------------------------------------------------------

def merge_sort(arr):
    """Return a new list containing the elements of arr in sorted order."""

    if len(arr) <= 1:
        return arr

    # Divide
    middle = len(arr) // 2
    left = merge_sort(arr[:middle])
    right = merge_sort(arr[middle:])

    # Conquer and combine
    return merge(left, right)


def merge(left, right):
    """Merge two sorted lists into one sorted list."""

    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])

    return result


# ---------------------------------------------------------
# QUICK SORT
# ---------------------------------------------------------

def quick_sort(arr, low=0, high=None):
    """Sort arr in place using Quick Sort."""

    if high is None:
        high = len(arr) - 1

    if low < high:
        pivot_index = partition(arr, low, high)

        quick_sort(arr, low, pivot_index - 1)
        quick_sort(arr, pivot_index + 1, high)

    return arr


def partition(arr, low, high):
    """
    Partition using the final element as the pivot.

    This simple pivot strategy intentionally demonstrates
    Quick Sort's poor behavior on already ordered data.
    """

    pivot = arr[high]
    i = low - 1

    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]

    arr[i + 1], arr[high] = arr[high], arr[i + 1]

    return i + 1


# ---------------------------------------------------------
# PERFORMANCE MEASUREMENT
# ---------------------------------------------------------

def measure_performance(sort_function, data):
    """
    Measure execution time and peak Python memory allocation.
    """

    test_data = data.copy()

    tracemalloc.start()

    start_time = time.perf_counter()

    result = sort_function(test_data)

    end_time = time.perf_counter()

    current_memory, peak_memory = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    execution_time = end_time - start_time
    peak_memory_kb = peak_memory / 1024

    # Verify that the algorithm actually sorted the data.
    assert result == sorted(data)

    return execution_time, peak_memory_kb


# ---------------------------------------------------------
# DATASET GENERATION
# ---------------------------------------------------------

def create_datasets(size):
    """Create sorted, reverse-sorted, and random datasets."""

    sorted_data = list(range(size))

    reverse_data = list(range(size, 0, -1))

    random_data = list(range(size))
    random.shuffle(random_data)

    return {
        "Sorted": sorted_data,
        "Reverse Sorted": reverse_data,
        "Random": random_data
    }


# ---------------------------------------------------------
# EXPERIMENT
# ---------------------------------------------------------

def run_experiment():

    # Moderate sizes prevent the intentionally worst-case
    # Quick Sort tests from taking excessively long.
    sizes = [100, 500, 1000, 2000]

    algorithms = {
        "Merge Sort": merge_sort,
        "Quick Sort": quick_sort
    }

    print(
        f"{'Algorithm':<12}"
        f"{'Dataset':<18}"
        f"{'Size':<8}"
        f"{'Time (sec)':<15}"
        f"{'Peak Memory (KB)':<18}"
    )

    print("-" * 71)

    for size in sizes:

        datasets = create_datasets(size)

        for dataset_name, data in datasets.items():

            for algorithm_name, algorithm in algorithms.items():

                execution_time, memory = measure_performance(
                    algorithm,
                    data
                )

                print(
                    f"{algorithm_name:<12}"
                    f"{dataset_name:<18}"
                    f"{size:<8}"
                    f"{execution_time:<15.6f}"
                    f"{memory:<18.2f}"
                )


if __name__ == "__main__":
    run_experiment()

Algorithm   Dataset           Size    Time (sec)     Peak Memory (KB)  
-----------------------------------------------------------------------
Merge Sort  Sorted            100     0.000781       2.53              
Quick Sort  Sorted            100     0.000819       0.09              
Merge Sort  Reverse Sorted    100     0.000520       1.98              
Quick Sort  Reverse Sorted    100     0.000601       0.09              
Merge Sort  Random            100     0.000635       1.79              
Quick Sort  Random            100     0.000251       0.09              
Merge Sort  Sorted            500     0.003499       10.38             
Quick Sort  Sorted            500     0.110598       15.27             
Merge Sort  Reverse Sorted    500     0.002465       9.80              
Quick Sort  Reverse Sorted    500     0.381270       16.78             
Merge Sort  Random            500     0.005740       8.27              
Quick Sort  Random            500     0.011910       1.21       